# 🏓 Pickleball Base Vision AI — Colab T4 GPU Training

This notebook trains the **Pickleball Ball Detector (Base Perception Model)** at high resolution ($1280\times 1280$) on NVIDIA Tesla T4 GPU with **ZERO hardcoded rules**.

### Runtime Setup
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
# Step 1: Verify NVIDIA GPU Runtime
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics YOLO
!pip install -q ultralytics

In [ ]:
# Step 3: Unzip Clean Dataset Archive
# Upload 'ball_clean_dataset.zip' via Colab left files panel or mount Google Drive
import os, zipfile

if os.path.exists("ball_clean_dataset.zip"):
    with zipfile.ZipFile("ball_clean_dataset.zip") as zf:
        zf.extractall(".")
    print("✅ Extracted ball_clean_dataset.zip")
else:
    print("⚠️ Please upload ball_clean_dataset.zip to the Colab files panel")

In [ ]:
# Step 4: Launch High-Resolution Training (T4 GPU)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

train_results = model.train(
    data="ball_clean/data.yaml",
    imgsz=1280,
    epochs=50,
    patience=12,
    batch=16,
    device=0,
    name="ball_colab_t4_1280",
    save=True,
    plots=True,
    verbose=True,
)

In [ ]:
# Step 5: Evaluate on Held-Out Test Split
val_metrics = model.val(
    data="ball_clean/data.yaml",
    split="test",
    imgsz=1280,
    device=0
)
print("Test Precision:", val_metrics.results_dict.get("metrics/precision(B)"))
print("Test Recall   :", val_metrics.results_dict.get("metrics/recall(B)"))
print("Test mAP@50   :", val_metrics.results_dict.get("metrics/mAP50(B)"))

In [ ]:
# Step 6: Download Best Checkpoint to Local Machine
from google.colab import files
import shutil

best_path = "runs/detect/ball_colab_t4_1280/weights/best.pt"
if os.path.exists(best_path):
    shutil.copyfile(best_path, "ball_best_t4.pt")
    files.download("ball_best_t4.pt")
    print("Downloading ball_best_t4.pt...")